# Automation for file Rearranging and Compiling

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# Meta data reading

In [2]:
# meta data reading for XPS data
def read_spectrum_meta_data(path, tools_to_read = ['H1', 'J4', 'K1', 'K4']):    
     #meta data for data
    df_md = pd.DataFrame(columns=['tool', 'wafer_id', 'datetime', 'path', 'lot_id'])
    
    for tool in tools_to_read:
        # create the path for the tool
        path_tool_data = path / tool
        # if no such tool directory exists, continue to the next tool
        if not path_tool_data.exists():
            continue
        paths = [Path(path) for path in path_tool_data.rglob('*.spe')]
        # if no data found, continue to the next tool
        if len(paths) == 0:
            continue
        # start reading the meta data and create columns
        # assume structure like:
        # RawData_USERNAME_2026-05-29_09384401\NTA000004.0H\NTA000004.23_20260517-152621\G94.NTA000004.23.01.20260517.152441.spe
        df_tmp = pd.DataFrame({'path': paths})
        df_tmp['wafer_id'] = df_tmp['path'].apply(lambda x: x.parents[0].name.split('_')[0])
        df_tmp['lot_id'] = df_tmp['path'].apply(lambda x: x.parents[1].name)
        df_tmp['datetime'] = df_tmp['path'].apply(lambda x: x.parents[0].name.split('_')[1])
        df_tmp['datetime'] = pd.to_datetime(df_tmp['datetime'], format='%Y%m%d-%H%M%S')
        df_tmp['tool'] = tool
        # append the read meta data
        df_md = pd.concat([df_md, df_tmp[df_md.columns]], ignore_index=True)

    # fix data type
    df_md = df_md.convert_dtypes()
    return df_md

In [ ]:
# Resovle script directory dynamically
# Assume the program is a python scirpt first, and then jupyter notebook if it fails (just for migration in the future)
try:
    path_scirpt = Path(__file__)
except:
    path_script = Path.cwd()
path_script

intermediary = 'example_data'
path_data = path_script / intermediary

df_md = read_spectrum_meta_data(path_data, tools_to_read = ['H1', 'J4', 'K1', 'K4'])

print(df_md.info())
df_md.head()


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tool      4 non-null      string        
 1   wafer_id  4 non-null      string        
 2   datetime  4 non-null      datetime64[us]
 3   path      4 non-null      object        
 4   lot_id    4 non-null      string        
dtypes: datetime64[us](1), object(1), string(3)
memory usage: 292.0+ bytes
None


,tool,wafer_id,datetime,path,lot_id
0,H1,NTA000004.23,2026-05-17 15:26:21,d:\gitlab\GGRD\Automation\example_data\H1\RawD...,NTA000004.0H
1,J4,NTA000001.16,2026-05-17 01:51:17,d:\gitlab\GGRD\Automation\example_data\J4\RawD...,NTA000001.0K
2,K1,NTA000003.22,2026-05-21 13:03:47,d:\gitlab\GGRD\Automation\example_data\K1\RawD...,NTA000003.0H
3,K4,NTA000002.19,2026-05-29 02:30:46,d:\gitlab\GGRD\Automation\example_data\K4\RawD...,NTA000002.0H


In [4]:
# path_script = path_script / 'example_data' / 'H1'
# path_script
# file_path = Path(r'RawData_USERNAME_2026-05-29_09384401\NTA000004.0H\NTA000004.23_20260517-152621\G94.NXB083468.01.01.20260517.152441.spe'.replace('\\', '/'), )
# file_path = path_script / file_path
# file_path.parent.mkdir(parents=True, exist_ok=True)
# file_path.touch()
# file_path.parent

In [5]:
# # examples to use shutil
# source = Path("D:/gitlab/GGRD/S2S/S2S_CNN.py")
# destination = Path("D:/gitlab/GGRD/S2S/backup_CNN.py")

# # Rename or Move
# source.rename(destination)

# # Copy a single file
# shutil.copy2(source, destination) # copy2 preserves metadata

# # Copy an entire directory
# shutil.copytree(Path("D:/gitlab/GGRD/S2S"), Path("D:/gitlab/GGRD/S2S_backup"))